# freeze-requires-grad — worked example 1: Freeze All Parameters Then Unfreeze a New Classification Head

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `freeze-requires-grad`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The standard transfer-learning recipe has two phases: first freeze every parameter in the pretrained backbone by setting `requires_grad = False`, then attach a new classification head (which defaults to `requires_grad = True`). Passing only the unfrozen head parameters to the optimizer ensures that the backbone weights don't move during fine-tuning. This preserves the pretrained features and only trains the task-specific output layer.

## Worked solution

We apply the freeze-and-swap pattern to a tiny three-layer network.

**Step 1 — freeze:** Iterate over `model.parameters()` and set each `p.requires_grad = False`. After this, no parameter receives a gradient.

**Step 2 — replace head:** Assign `model.fc = nn.Linear(hidden_dim, n_classes)`. A newly constructed `nn.Linear` always starts with `requires_grad = True` on both its weight and bias tensors.

**Step 3 — collect trainable params:** Build `[p for p in model.parameters() if p.requires_grad]`. This list contains exactly the new head's weight and bias — 2 tensors total.

**Step 4 — verify via backward:** Run a forward pass, compute a loss, call `.backward()`. Only the head's weight and bias should accumulate gradients; the backbone params should have `grad = None`.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(30)

# Toy pretrained model
class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(8, 16)
        self.layer2 = nn.Linear(16, 8)
        self.fc     = nn.Linear(8, 100)  # old head
    def forward(self, x):
        return self.fc(t.relu(self.layer2(t.relu(self.layer1(x)))))

model = ToyNet()
n_classes = 5

# Step 1: freeze everything
for p in model.parameters():
    p.requires_grad = False
assert all(not p.requires_grad for p in model.parameters())

# Step 2: replace head
model.fc = nn.Linear(8, n_classes)
assert model.fc.weight.requires_grad
assert model.fc.bias.requires_grad

# Step 3: collect trainable
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable tensors: {len(trainable)} (expected 2: new fc weight + bias)")
assert len(trainable) == 2
new_fc_params = list(model.fc.parameters())
assert any(p is trainable[0] for p in new_fc_params)

# Step 4: backward pass — only head grads
x = t.randn(4, 8)
loss = model(x).sum()
loss.backward()
assert model.fc.weight.grad is not None
assert model.fc.bias.grad is not None
assert model.layer1.weight.grad is None  # backbone still frozen
print("Backbone grads are None, head grads exist — freeze + swap works.")